In [48]:
import numpy as np
import pandas as pd
from collections import defaultdict, deque
import heapq
import csv

In [49]:
folder = "graphs/"

In [83]:
class Graph(object):

    """
    Initiate the graph with an adjancency matrix read from a csv file.

    Parameters
    ----------
    file : str
        The name of the csv file containing the adjacency matrix.
    """
    def __init__(self, file, directed=False):
        self.directed = directed
        df = pd.read_csv(folder+file, sep=';', header=0, engine='python')
        matrix = df.values
        
        self.n = len(matrix)

        self.init_AdjList(matrix)

    """
    Initialize the adjacency list.

    Parameters
    ----------
    matrix : list of list of int
        The adjacency matrix of the graph.
    """
    def init_AdjList(self,matrix):
        self.adjList = defaultdict(list)

        for i in range(self.n):
            for j in range(self.n):
                if matrix[i][j] != 0:
                    self.adjList[i].append((j, matrix[i][j]))

        self.validateGraph()

    def validateGraph(self):
        if not self.directed:
            for v, adj in self.adjList.items():
                for u, _ in adj:
                    if v not in [x for x, _ in self.adjList[u]]:
                        raise ValueError(f"Edge ({v}, {u}) is not symmetric.")
        
        #handshake lemma: the number of vertices with odd degree must be even
        #if sum(1 for d in self.nodesDegrees() if d % 2 == 1) % 2 != 0:
         #   raise ValueError("Invalid graph: odd number of odd-degree vertices.")

    """
    Return the number of vertices in the graph.

    Returns
    -------
    int
        The number of vertices in the graph.
    """
    def __len__(self):
        return self.n

    """
    Add an edge between vertices u and v.
    
    Parameters
    ----------
    u : int
        The first vertex.
    v : int
        The second vertex.
    """
    def add_edge(self, u, v, w):

        if not (0 <= u < self.n and 0 <= v < self.n):
            raise ValueError("node out of bounds")

        self.adjList[u].append((v,w))

        if not self.directed:
            self.adjList[v].append((u,w))

        self.validateGraph()

    """
    Remove the edge between vertices u and v.
    
    Parameters
    ----------
    u : int
        The first vertex.
    v : int
        The second vertex.
    """
    def remove_edge(self, u, v):
        self.adjList[u] = [(x, w) for (x, w) in self.adjList[u] if x != v]

        if not self.directed:
            self.adjList[v] = [(x, w) for (x, w) in self.adjList[v] if x != u]

        self.validateGraph()
    
    """
    Return the number of Vertices in the graph.

    Returns
    -------

    int
        The number of vertices in the graph.
    """
    def numNodes(self):
        return self.n
    
    """
    Return the number of edges in the graph.
    """
    def numEdges(self):
        return(int (sum(self.nodesDegrees()) / 2))
    
    """
    Return a dictionary with the degree of each vertex in the graph.
    
    Returns:
    -------
    dict
        A dictionary with the degree of each vertex in the graph.
    """
    def nodesDegrees(self):
        return [self.nodeDegree(v) for v in range(self.n)]
    
    def nodeDegree(self, v):
        deg = len(self.adjList[v])

        if v in self.adjList[v]:
            deg += self.adjList[v].count(v)

        return deg

    """
    Return True if the graph has parallel edges, False otherwise.
    
    Returns:
    -------
    bool        
        True if the graph has parallel edges, False otherwise. 
    """
    def hasParallelEdges(self):
        if any(len(adj) != len(set(adj)) for adj in self.adjList.values()):
            return True
        return False
    
    """
    Return a dictionary with the parallel edges in the graph.

    Returns:
    -------
    dict
        A dictionary with the parallel edges in the graph.
    """
    def parallelEdges(self):
        parallels = {}

        for v in range(self.n):
            seen = set()
            duplicates = []

            for u in self.adjList[v]:
                if u in seen:
                    duplicates.append(u)
                else:
                    seen.add(u)

            if duplicates:
                parallels[v] = duplicates

        return parallels
    
    def hasLoopEdges(self):
        return self.loopEdges() != {}

    def loopEdges(self):
        loopEdges = {
            v: adj 
            for v, adj in self.adjList.items() 
            if v in adj
            }
        return loopEdges
    
    def printAdjList(self):
        for v in range(self.n):
            print(f"Vertex {v}: {self.adjList[v]}")

    def isSimple(self):
         return not self.hasParallelEdges() and not self.hasLoopEdges()
    
    def isNull(self):
        if self.numEdges() == 0:
            return True
        return False
    
    def isRegular(self):
        degrees = self.nodesDegrees()
        return len(set(degrees)) == 1
    
    def isolatedNodes(self):
        return self.nodesOfDegree(0)
    
    def pendingNodes(self):
        return self.nodesOfDegree(1)
    
    def nodesOfDegree(self, degree):
        return [v for v in range(self.n) if self.nodeDegree(v) == degree]
        
    def isComplete(self):
        return self.numEdges() == (self.n - 1) * self.n // 2 

    def isConnected(self):  
        
        bfs_visited, _, _ = self.bfs(0)

        return len(bfs_visited) == self.n

    def isCircular(self):

        if not self.isSimple():
            return False
        
        if not self.isConnected():
            return False
        
        if self.numEdges() != self.n:
            return False

        return all(d == 2 for d in self.nodesDegrees())
    
    """
    Breadth First Search (BFS) Implementation

    Parameters
    ----------
    root : int
        The starting vertex for the BFS.
    
    Returns
    -------
    visited : set
        A set of visited vertices.
    parent : dict
        A dictionary mapping each vertex to its parent in the BFS tree.
    dist : dict
        A dictionary mapping each vertex to its distance from the root vertex.
    """
    def bfs(self, root):
        if self.n == 0:
            return set(), {}, {}

        visited = set()
        parent = {root: None}
        dist = {root: 0}

        queue = deque([root])
    
        while len(queue) > 0:
            v = queue.popleft()
            
            if v not in visited:
                visited.add(v)

                for w, _ in self.adjList[v]:
                    if w not in dist:
                        dist[w] = dist[v] + 1
                        parent[w] = v
                        queue.append(w)
        
        return visited, parent, dist
    

    """
    Depth First Search (DFS) Implementation

    Parameters
    ----------
    root : int
        The starting vertex for the DFS.
    
    Returns
    -------
    visited : set
        A set of visited vertices.
    parent : dict
        A dictionary mapping each vertex to its parent in the DFS tree.
    dist : dict
        A dictionary mapping each vertex to its distance from the root vertex.
    """
    def dfs (self, root):
        visited = set()
        parent = {root: None}
        dist = {root: 0}
        has_cycle = False

        def _dfs_recursive(u):
            visited.add(u)

            for v in self.adjList[u]:
                if v in visited and parent[u] != v:
                    nonlocal has_cycle
                    has_cycle = True
                if v not in visited:
                    parent[v] = u
                    dist[v] = dist[u] + 1
                    _dfs_recursive(v)
        _dfs_recursive(root)
        return visited, parent, dist, has_cycle


    def shortest_path(self, root):

        min_heap = [(0,root)]
        parent = {root: None}
        dist = {node: float('inf') for node in range(self.n)}
        dist[root] = 0

        while min_heap:
            curr_dist, v = heapq.heappop(min_heap)

            if curr_dist > dist[v]:
                continue

            for w, weight in self.adjList[v]:
                new_dist = dist[v] + weight
                if new_dist < dist[w]:
                    dist[w] = new_dist
                    parent[w] = v
                    heapq.heappush(min_heap,(new_dist, w))

        return dist, parent

    """
    Return True if the graph is a tree, False otherwise.
    Returns
    -------
    bool
        True if the graph is a tree, False otherwise.
    """
    def isTree(self):
        if not self.isConnected():
            return False
        
        if self.numEdges() != self.n - 1:
            return False
        return True


    def has_cycle(self):
        _, _, _, has_cycle = self.dfs(0)
        return has_cycle
    """
    INCOMPLETE
    Return True if the graph and another graph G2 are isomorphs, False otherwise.

    Parameters
    ----------
    G2 : Graph
        The graph to compare with.
    
    Returns
    -------
    bool
        True if the graph and G2 are isomorphs, False otherwise.
    """
    def areIsomorphs(self, G2):
        if self.numNodes() != G2.numNodes() or self.numEdges() != G2.numEdges():
            return False
    
    def areComplementary(self, G2):
        return( "not implemented")

In [84]:
a = defaultdict(list)
a[1]=1
a[2]=3
a["a"]=4

for i in a:
    print(i)

1
2
a


In [85]:
n3e2 = Graph("n3e2(in).csv")
print("n =", n3e2.numNodes(), "\te =", n3e2.numEdges() )

n = 3 	e = 2


In [86]:
print("loop {}".format(n3e2.loopEdges()))
print("paralel {}".format(n3e2.parallelEdges()))
n3e2.adjList.items()

loop {}
paralel {}


dict_items([(0, [(1, np.int64(1)), (2, np.int64(1))]), (1, [(0, np.int64(1))]), (2, [(0, np.int64(1))])])

In [87]:
n3e2.dfs(1)
n3e2.bfs(1)

({0, 1, 2}, {1: None, 0: 1, 2: 0}, {1: 0, 0: 1, 2: 2})

In [88]:
K5 = Graph("K5(in).csv")
print("n =", K5.numNodes(), "\te =", K5.numEdges() )

n = 5 	e = 10


In [89]:
K5.printAdjList()
K5.nodesDegrees()
K5.isCircular()

Vertex 0: [(1, np.int64(1)), (2, np.int64(1)), (3, np.int64(1)), (4, np.int64(1))]
Vertex 1: [(0, np.int64(1)), (2, np.int64(1)), (3, np.int64(1)), (4, np.int64(1))]
Vertex 2: [(0, np.int64(1)), (1, np.int64(1)), (3, np.int64(1)), (4, np.int64(1))]
Vertex 3: [(0, np.int64(1)), (1, np.int64(1)), (2, np.int64(1)), (4, np.int64(1))]
Vertex 4: [(0, np.int64(1)), (1, np.int64(1)), (2, np.int64(1)), (3, np.int64(1))]


False

In [90]:
tree = Graph("tree.csv")

In [91]:
tree.printAdjList()

Vertex 0: [(1, np.int64(1)), (2, np.int64(1)), (3, np.int64(1))]
Vertex 1: [(0, np.int64(1))]
Vertex 2: [(0, np.int64(1))]
Vertex 3: [(0, np.int64(1)), (4, np.int64(1)), (5, np.int64(1))]
Vertex 4: [(3, np.int64(1))]
Vertex 5: [(3, np.int64(1))]


In [92]:
tree.isTree()

True

In [93]:
tree.add_edge(4,5,2)

In [94]:
tree.printAdjList()

Vertex 0: [(1, np.int64(1)), (2, np.int64(1)), (3, np.int64(1))]
Vertex 1: [(0, np.int64(1))]
Vertex 2: [(0, np.int64(1))]
Vertex 3: [(0, np.int64(1)), (4, np.int64(1)), (5, np.int64(1))]
Vertex 4: [(3, np.int64(1)), (5, 2)]
Vertex 5: [(3, np.int64(1)), (4, 2)]


In [95]:
tree.has_cycle()

False

In [96]:
tree.isTree()

False

In [97]:
tree.isCircular()

False

In [98]:
tree.printAdjList()

Vertex 0: [(1, np.int64(1)), (2, np.int64(1)), (3, np.int64(1))]
Vertex 1: [(0, np.int64(1))]
Vertex 2: [(0, np.int64(1))]
Vertex 3: [(0, np.int64(1)), (4, np.int64(1)), (5, np.int64(1))]
Vertex 4: [(3, np.int64(1)), (5, 2)]
Vertex 5: [(3, np.int64(1)), (4, 2)]


In [99]:
tree.remove_edge(4,5)

In [100]:
tree.has_cycle()

False

In [101]:
v_dfs, p_dfs, d_dfs = tree.dfs(1)
v_bfs, p_bfs, d_bfs = tree.bfs(1)

print(f"DFS -> {p_dfs} e BFS -> {p_bfs}")


ValueError: too many values to unpack (expected 3, got 4)

In [102]:
print("Numero de arestas:", tree.numEdges())

Numero de arestas: 5


In [103]:
null = Graph("null.csv")

In [104]:
null.printAdjList()

Vertex 0: []
Vertex 1: []
Vertex 2: []
Vertex 3: []


In [105]:
null.isSimple()

True

In [106]:
C4 = Graph("C4.csv")
C4.printAdjList()

Vertex 0: [(1, np.float64(1.0)), (3, np.float64(1.0))]
Vertex 1: [(0, np.float64(1.0)), (2, np.float64(1.0))]
Vertex 2: [(1, np.float64(1.0)), (3, np.float64(1.0))]
Vertex 3: [(0, np.float64(1.0)), (2, np.float64(1.0))]


In [107]:
C4.isCircular()

True

In [108]:
C4.has_cycle()

False

In [109]:
C2 = Graph("C2.csv")
C2.printAdjList()
C2.isCircular()

Vertex 0: [(1, np.float64(1.0))]
Vertex 1: [(0, np.float64(1.0)), (2, np.float64(1.0))]
Vertex 2: [(1, np.float64(1.0))]


False

In [111]:
C1 = Graph("C1.csv")
C1.printAdjList()
C1.isCircular()
C1.isSimple()

Vertex 0: [(0, np.float64(1.0))]


True